In [ ]:
# =============================================================================
# 1. Environment and submission mode
#
# Detect whether this is a real competition rerun (which minimises diagnostics),
# set the framework's environment flags, and put the CUDA libraries on the
# linker path.
# =============================================================================

import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# In submission, disable the periodic JSON/HTML diagnostics writes and per-frame logging.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")


# --- FIX 1 (new) -------------------------------------------------------------
# Surface a GPU/wheelhouse mismatch loudly instead of letting vLLM fail silently
# later. The bundled wheelhouse (arc3-vllm-h100-wheelhouse-v3) is built for H100;
# ARC-AGI-3's free accelerator pool is RTX 6000 (g4-standard-48). If the wrong
# GPU is attached, this print is the only thing that will tell you why setup
# failed, since minimal diagnostics suppresses almost everything else.
try:
    gpu_name = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
    ).strip().split("\n")[0]
except Exception as exc:
    gpu_name = f"<nvidia-smi failed: {exc}>"
print(f"taaf.kaggle: attached GPU = {gpu_name}")
if "H100" not in gpu_name and TRUE_SUBMISSION:
    print(
        f"taaf.kaggle: WARNING - wheelhouse is H100-built but attached GPU is '{gpu_name}'. "
        "If vLLM fails to load kernels below, this is why."
    )
# --- end FIX 1 -----------------------------------------------------------------


# =============================================================================
# 2. Install the ARC runtime
#
# Install `arc-agi` from the offline competition wheelhouse (the Kaggle
# submission environment has no internet).
# =============================================================================

# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)


# =============================================================================
# 3. Locate the source bundle
#
# Find the uploaded TAAF source dataset by its marker file, and record where
# Kaggle mounted every attached input so setup commands and the solver can
# find them.
# =============================================================================

# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["jeroencottaar/taaf-kaggle-source-share", "driessmit1/arc3-vllm-h100-wheelhouse-v3", "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")


# =============================================================================
# 4. Import the bundled source and run solver setup
#
# Put the snapshotted repositories on the path (this process and any child
# processes), then run the solver's setup commands - installing wheels,
# fetching model weights, and so on.
# =============================================================================

# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# --- FIX 2 (changed) -----------------------------------------------------------
# Original used `subprocess.run(..., check=True)`: a single transient failure
# (e.g. vLLM server failing to bind on first try) killed the whole notebook
# before the benchmark ever loaded, producing a hard-zero submission. Retry
# each setup command up to 3 times with backoff; only raise after all 3 fail.
env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    result = None
    for attempt in range(1, 4):
        print(f"taaf.kaggle: setup command (attempt {attempt}/3): {command}", flush=True)
        result = subprocess.run(command, shell=True, cwd=WORKING_DIR, env=env)
        if result.returncode == 0:
            break
        print(f"taaf.kaggle: setup command failed (exit {result.returncode}), retrying...", flush=True)
        time.sleep(10 * attempt)
    else:
        raise subprocess.CalledProcessError(result.returncode, command)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)
# --- end FIX 2 -------------------------------------------------------------

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)


# =============================================================================
# 5. Load the benchmark
#
# Unpickle the deployment target and the benchmark, stamping the real
# submission state onto the target and pointing the benchmark's outputs at
# the Kaggle working directory.
# =============================================================================

# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR


# =============================================================================
# 6. Customization hook
#
# Optional: tweak `bm`, `bm.games`, or `bm.solver` here before the run starts
# - the safe place for one-off experiments once the deployed bundle has
# loaded.
# =============================================================================

# Make one-off changes to `bm`, `bm.games`, or `bm.solver` here before the run starts.
# Example:
# bm.label = f"{bm.label}-debug"


# =============================================================================
# 6A. Three-member agent council augmentation
#
# This keeps the original deployed solver as the primary action agent and adds
# two advisory agents at the runtime boundary. The council is injected into
# local analyzer requests without assuming private class names inside the source
# bundle. If the hidden solver already has prompt fields, those fields are also
# extended in-place.
# =============================================================================

from dataclasses import asdict, dataclass
from typing import Any, Iterable
import copy
import functools
import inspect


@dataclass(frozen=True)
class CouncilAgentSpec:
    name: str
    role: str
    mandate: str
    veto_rule: str
    tie_break_weight: float


COUNCIL_AGENT_SPECS = [
    CouncilAgentSpec(
        name="Primary Solver Agent",
        role="Action proposer and final interface keeper",
        mandate=(
            "Use the deployed solver's original policy, game memory, retrieved patterns, and action schema. "
            "Never change the required response format. Propose the next legal action or click exactly as the solver normally would."
        ),
        veto_rule="Cannot override a hard legality failure raised by either advisor.",
        tie_break_weight=0.51,
    ),
    CouncilAgentSpec(
        name="Transition Auditor Agent",
        role="Frame-delta verifier",
        mandate=(
            "Compare the latest frame with prior frames. Identify the controllable object, moved objects, locked objects, doors, keys, goals, "
            "teleports, hazards, and irreversible state changes. Reject hallucinated progress, repeated no-op moves, and actions that contradict "
            "visible state transitions. Prefer actions whose outcome can be locally verified on the next frame."
        ),
        veto_rule=(
            "Veto an action when it is illegal, repeats a detected no-op, moves into an observed blocker, undoes required progress, "
            "or depends on an unobserved object/state."
        ),
        tie_break_weight=0.27,
    ),
    CouncilAgentSpec(
        name="Efficiency Judge Agent",
        role="RHAE score and route-efficiency critic",
        mandate=(
            "Minimize total actions because the ARC-AGI-3 score rewards action efficiency. Prefer shortest verified routes, cached exact moves, "
            "level-completion actions, and low-branching plans. Detect loops from recent frame/action history and force an unexplored legal action "
            "when the primary proposal is cycling."
        ),
        veto_rule=(
            "Veto an action when a shorter verified action exists, when the move repeats a recent cycle without new information, "
            "or when it spends actions after a level-completion condition is visible."
        ),
        tie_break_weight=0.22,
    ),
]

COUNCIL_INSTRUCTION_BLOCK = """

[ARC-AGI-3 THREE-MEMBER COUNCIL MODE]
You are still bound to the existing solver contract and must return exactly the same output format requested by the original prompt.
Do not reveal council deliberation, role names, hidden reasoning, analysis text, or extra commentary in the final answer.

Council members:
1. Primary Solver Agent — keeps the deployed policy and required action schema.
2. Transition Auditor Agent — verifies frame deltas, object permanence, blockers, keys, goals, hazards, and hallucinated progress.
3. Efficiency Judge Agent — minimizes action count for RHAE scoring, avoids loops, and prefers shortest verified routes.

Decision protocol:
- Start from the Primary Solver Agent's candidate action.
- Transition Auditor may veto illegal, impossible, no-op, blocker-colliding, or hallucinated-progress actions.
- Efficiency Judge may veto looped or unnecessarily long actions when a shorter verified action is available.
- If there is disagreement, choose the legal action with the best combination of visible progress, shortest expected route, and lowest loop risk.
- If uncertainty remains, choose the action that creates new information while preserving progress.
- Final output must be only the original action/click/JSON schema expected by the solver; no council transcript.
[END COUNCIL MODE]
""".strip()


def _persist_council_env() -> None:
    """Publish council configuration to this process and to setup env for child processes."""
    council_json = json.dumps([asdict(agent) for agent in COUNCIL_AGENT_SPECS], sort_keys=True)
    env_updates = {
        "TAAF_COUNCIL_ENABLED": "1",
        "TAAF_COUNCIL_MODE": "primary_plus_two_advisors",
        "TAAF_COUNCIL_AGENT_SPECS": council_json,
        "LOCAL_ANALYZER_COUNCIL_AGENT_SPECS": council_json,
        "LOCAL_ANALYZER_COUNCIL_INJECTION": COUNCIL_INSTRUCTION_BLOCK,
    }
    os.environ.update(env_updates)
    try:
        existing_setup_env = json.loads(SETUP_ENV_PATH.read_text(encoding="utf-8")) if SETUP_ENV_PATH.exists() else {}
        if isinstance(existing_setup_env, dict):
            existing_setup_env.update(env_updates)
            SETUP_ENV_PATH.write_text(json.dumps(existing_setup_env, indent=2, sort_keys=True), encoding="utf-8")
    except Exception as exc:
        print(f"taaf.kaggle: council env persistence skipped: {exc}", flush=True)


def _append_once(text: str, block: str) -> str:
    text = "" if text is None else str(text)
    return text if block in text else f"{text}\n\n{block}".strip()


def _message_text_content_has_block(content: Any) -> bool:
    if isinstance(content, str):
        return COUNCIL_INSTRUCTION_BLOCK in content
    if isinstance(content, list):
        for part in content:
            if isinstance(part, dict) and COUNCIL_INSTRUCTION_BLOCK in str(part.get("text", "")):
                return True
    return False


def _append_block_to_message_content(content: Any) -> Any:
    if isinstance(content, str):
        return _append_once(content, COUNCIL_INSTRUCTION_BLOCK)
    if isinstance(content, list):
        updated = copy.deepcopy(content)
        for part in updated:
            if isinstance(part, dict) and part.get("type") in {"text", "input_text"} and "text" in part:
                part["text"] = _append_once(str(part.get("text", "")), COUNCIL_INSTRUCTION_BLOCK)
                return updated
        updated.insert(0, {"type": "text", "text": COUNCIL_INSTRUCTION_BLOCK})
        return updated
    return _append_once(str(content), COUNCIL_INSTRUCTION_BLOCK)


def _inject_council_into_messages(messages: Any) -> Any:
    """Return a copy of OpenAI-style messages with a single council instruction injected."""
    if not isinstance(messages, list):
        return messages
    cloned = copy.deepcopy(messages)
    for message in cloned:
        if isinstance(message, dict) and _message_text_content_has_block(message.get("content")):
            return cloned
    for preferred_role in ("system", "developer", "user"):
        for message in cloned:
            if isinstance(message, dict) and str(message.get("role", "")).lower() == preferred_role:
                message["content"] = _append_block_to_message_content(message.get("content", ""))
                return cloned
    cloned.insert(0, {"role": "system", "content": COUNCIL_INSTRUCTION_BLOCK})
    return cloned


def _inject_council_into_payload(payload: Any) -> Any:
    if not isinstance(payload, dict):
        return payload
    if "messages" not in payload:
        return payload
    updated = dict(payload)
    updated["messages"] = _inject_council_into_messages(updated.get("messages"))
    return updated


def _looks_like_chat_url(url: Any) -> bool:
    text = str(url).lower()
    return "/chat/completions" in text or text.endswith("chat/completions")


def _install_requests_council_patch() -> None:
    try:
        import requests
        target_cls = requests.sessions.Session
        if getattr(target_cls.request, "_taaf_council_patched", False):
            return
        original_request = target_cls.request

        @functools.wraps(original_request)
        def council_request(self, method, url, **kwargs):
            try:
                if _looks_like_chat_url(url) and isinstance(kwargs.get("json"), dict):
                    kwargs = dict(kwargs)
                    kwargs["json"] = _inject_council_into_payload(kwargs["json"])
            except Exception as exc:
                print(f"taaf.kaggle: requests council injection skipped: {exc}", flush=True)
            return original_request(self, method, url, **kwargs)

        council_request._taaf_council_patched = True
        council_request._taaf_council_original = original_request
        target_cls.request = council_request
    except Exception as exc:
        print(f"taaf.kaggle: requests council patch unavailable: {exc}", flush=True)


def _install_httpx_council_patch() -> None:
    try:
        import httpx

        if not getattr(httpx.Client.request, "_taaf_council_patched", False):
            original_client_request = httpx.Client.request

            @functools.wraps(original_client_request)
            def council_client_request(self, method, url, **kwargs):
                try:
                    if _looks_like_chat_url(url) and isinstance(kwargs.get("json"), dict):
                        kwargs = dict(kwargs)
                        kwargs["json"] = _inject_council_into_payload(kwargs["json"])
                except Exception as exc:
                    print(f"taaf.kaggle: httpx client council injection skipped: {exc}", flush=True)
                return original_client_request(self, method, url, **kwargs)

            council_client_request._taaf_council_patched = True
            council_client_request._taaf_council_original = original_client_request
            httpx.Client.request = council_client_request

        if not getattr(httpx.AsyncClient.request, "_taaf_council_patched", False):
            original_async_request = httpx.AsyncClient.request

            @functools.wraps(original_async_request)
            async def council_async_request(self, method, url, **kwargs):
                try:
                    if _looks_like_chat_url(url) and isinstance(kwargs.get("json"), dict):
                        kwargs = dict(kwargs)
                        kwargs["json"] = _inject_council_into_payload(kwargs["json"])
                except Exception as exc:
                    print(f"taaf.kaggle: httpx async council injection skipped: {exc}", flush=True)
                return await original_async_request(self, method, url, **kwargs)

            council_async_request._taaf_council_patched = True
            council_async_request._taaf_council_original = original_async_request
            httpx.AsyncClient.request = council_async_request
    except Exception as exc:
        print(f"taaf.kaggle: httpx council patch unavailable: {exc}", flush=True)


def _install_openai_council_patch() -> None:
    """Patch OpenAI SDK chat completions if the hidden solver uses the SDK directly."""
    try:
        from openai.resources.chat.completions import AsyncCompletions, Completions

        if not getattr(Completions.create, "_taaf_council_patched", False):
            original_create = Completions.create

            @functools.wraps(original_create)
            def council_create(self, *args, **kwargs):
                try:
                    if isinstance(kwargs.get("messages"), list):
                        kwargs = dict(kwargs)
                        kwargs["messages"] = _inject_council_into_messages(kwargs["messages"])
                except Exception as exc:
                    print(f"taaf.kaggle: openai council injection skipped: {exc}", flush=True)
                return original_create(self, *args, **kwargs)

            council_create._taaf_council_patched = True
            council_create._taaf_council_original = original_create
            Completions.create = council_create

        if not getattr(AsyncCompletions.create, "_taaf_council_patched", False):
            original_async_create = AsyncCompletions.create

            @functools.wraps(original_async_create)
            async def council_async_create(self, *args, **kwargs):
                try:
                    if isinstance(kwargs.get("messages"), list):
                        kwargs = dict(kwargs)
                        kwargs["messages"] = _inject_council_into_messages(kwargs["messages"])
                except Exception as exc:
                    print(f"taaf.kaggle: async openai council injection skipped: {exc}", flush=True)
                return await original_async_create(self, *args, **kwargs)

            council_async_create._taaf_council_patched = True
            council_async_create._taaf_council_original = original_async_create
            AsyncCompletions.create = council_async_create
    except Exception as exc:
        print(f"taaf.kaggle: openai council patch unavailable: {exc}", flush=True)


def _extend_solver_prompt_attributes(solver: Any) -> list[str]:
    """Best-effort extension for visible prompt attributes on the restored solver."""
    prompt_attr_names = {
        "system_prompt", "prompt", "base_prompt", "instruction", "instructions",
        "analyzer_prompt", "analysis_prompt", "policy_prompt", "developer_prompt",
        "decision_prompt", "action_prompt", "agent_prompt",
    }
    changed = []
    for attr in sorted(prompt_attr_names):
        if hasattr(solver, attr):
            try:
                value = getattr(solver, attr)
                if isinstance(value, str):
                    setattr(solver, attr, _append_once(value, COUNCIL_INSTRUCTION_BLOCK))
                    changed.append(attr)
            except Exception:
                pass
    try:
        if hasattr(solver, "__dict__"):
            for attr, value in list(vars(solver).items()):
                if isinstance(attr, str) and "prompt" in attr.lower() and isinstance(value, str):
                    setattr(solver, attr, _append_once(value, COUNCIL_INSTRUCTION_BLOCK))
                    if attr not in changed:
                        changed.append(attr)
    except Exception:
        pass
    return changed


def _attach_council_to_solver(solver: Any) -> Any:
    _persist_council_env()
    _install_requests_council_patch()
    _install_httpx_council_patch()
    _install_openai_council_patch()
    changed_prompt_attrs = _extend_solver_prompt_attributes(solver)
    try:
        setattr(solver, "council_agent_specs", COUNCIL_AGENT_SPECS)
        setattr(solver, "council_instruction_block", COUNCIL_INSTRUCTION_BLOCK)
        setattr(solver, "council_mode", "primary_plus_transition_auditor_plus_efficiency_judge")
    except Exception:
        pass
    try:
        if hasattr(solver, "label") and "council" not in str(getattr(solver, "label", "")).lower():
            solver.label = f"{solver.label}-council3"
        if "council" not in str(getattr(bm, "label", "")).lower():
            bm.label = f"{bm.label}-council3"
    except Exception:
        pass
    (WORKING_DIR / "council_agents.json").write_text(
        json.dumps(
            {
                "enabled": True,
                "mode": "primary_plus_two_advisors",
                "agents": [asdict(agent) for agent in COUNCIL_AGENT_SPECS],
                "patched_prompt_attrs": changed_prompt_attrs,
                "solver_type": type(solver).__name__,
                "solver_module": type(solver).__module__,
            },
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )
    print(
        "taaf.kaggle: installed 3-member council "
        "(primary solver + Transition Auditor + Efficiency Judge)",
        flush=True,
    )
    if changed_prompt_attrs:
        print(f"taaf.kaggle: council extended prompt attrs = {changed_prompt_attrs}", flush=True)
    return solver


bm.solver = _attach_council_to_solver(bm.solver)



# =============================================================================
# 7. Run the benchmark
#
# In a real competition rerun (`KAGGLE_IS_COMPETITION_RERUN`), wait for the
# Kaggle gateway and play the live competition Arcade. Otherwise - an
# interactive "Save & Run" - play the competition's bundled environment files
# offline, with no gateway required, so the notebook runs end-to-end without
# a submission. Teardown commands run afterward even if the run raises.
# =============================================================================

# Build the live competition game list from the gateway's available environments.
def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# Build the offline game list from the competition's bundled environment files.
def _offline_games(env_dir: str):
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# The gateway can take a while to come up; poll until it answers.
def _wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")


# Print the run preamble and persist the launcher's git status for diagnostics.
print((BUNDLE_DIR / "preamble.txt").read_text())
(WORKING_DIR / "git_status.txt").write_text((BUNDLE_DIR / "git_status.txt").read_text())

# arc_agi reads RECORDINGS_DIR and ARC_API_KEY from env (ArcadeSpec carries neither); operation
# mode, environments dir, and base url are all passed explicitly via the spec, so no env is needed.
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

if TRUE_SUBMISSION:
    # Real submission: play the live competition Arcade served by the Kaggle gateway.
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    # The gateway boots asynchronously; wait before swapping in its game list.
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    bm.games = _competition_games()
else:
    # Interactive run: play the bundled competition environments offline (no gateway).
    # The competition's environment files ship alongside the wheelhouse in the competition dataset.
    competition_env_files = str(Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels").parent / "environment_files")
    bm.games = _offline_games(competition_env_files)

# NOTE ON n_passes: raising this to retry stochastic rollouts only helps if
# bm.run()'s scoring takes the BEST score per game across passes rather than
# averaging them - and it costs Nx runtime inside the 9h Kaggle ceiling. Left
# at 1 here because that scoring behaviour lives inside taaf.benchmark, which
# isn't visible in this harness; confirm it there before changing this value.
bm.n_passes = 1
bm.game_weights = None

# --- FIX 3 (changed) -------------------------------------------------------
# Original gated the soft deadline entirely behind `if not TRUE_SUBMISSION`,
# so a real competition rerun had NO graceful exit at all - a stuck game
# could burn the full 9h Kaggle ceiling, the process gets killed, and neither
# teardown_commands nor a submission.parquet ever run. Both branches now get
# a soft_end: interactive runs keep the original margin, and real submissions
# get a margin sized to leave enough time for teardown + parquet write.
KAGGLE_HARD_LIMIT_S = 9 * 3600
budget = float(getattr(target, "max_runtime_s", 0.0) or 0.0)
if TRUE_SUBMISSION:
    budget = budget or KAGGLE_HARD_LIMIT_S
    margin = min(900.0, budget * 0.1)  # bail early so teardown + parquet write actually finish
else:
    margin = min(600.0, budget / 2) if budget > 0 else 0.0
soft_end = (
    datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(seconds=budget - margin)
    if budget > 0 else None
)
print(f"taaf.kaggle: soft_end = {soft_end} (budget={budget}s, margin={margin}s)")
# --- end FIX 3 ---------------------------------------------------------------

# Play the benchmark; teardown commands run even if the run raises.
try:
    result = await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=TRUE_SUBMISSION)

    # --- FIX 4 (new) ---------------------------------------------------------
    # Under TAAF_MINIMAL_DIAGNOSTICS, virtually nothing gets written for a real
    # submission, so a bad score is unexplainable after the fact - no way to
    # tell setup failure from solver failure from a hard game rotation. Write
    # a tiny best-effort summary regardless of mode. `result`'s actual shape
    # is not visible in this harness (lives in taaf.benchmark); repr() is a
    # deliberately conservative fallback that can't itself raise on unknown
    # attributes. Confirm the real return type there and tighten this once known.
    try:
        summary = {
            "true_submission": TRUE_SUBMISSION,
            "result_repr": repr(result)[:2000],
            "games": [getattr(g, "env_name", repr(g)) for g in getattr(bm, "games", [])],
            "n_passes": bm.n_passes,
        }
        (WORKING_DIR / "run_summary.json").write_text(json.dumps(summary, indent=2, default=str))
        print("taaf.kaggle: wrote run_summary.json")
    except Exception as diag_exc:
        print(f"taaf.kaggle: summary dump skipped: {diag_exc}")
    # --- end FIX 4 ---------------------------------------------------------

    if not TRUE_SUBMISSION:
        # An offline run isn't scored, but Kaggle still expects a submission.parquet output.
        import pandas as pd

        pd.DataFrame(
            [["1_0", "1", True, 1]],
            columns=["row_id", "game_id", "end_of_game", "score"],
        ).to_parquet(WORKING_DIR / "submission.parquet", index=False)
finally:
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print(f"taaf.kaggle: teardown command: {command}", flush=True)
        subprocess.run(command, shell=True, check=False, cwd=WORKING_DIR, env=_command_env())


# =============================================================================
# 8. Show the diagnostics
#
# A non-submission run writes `diagnostics.html` to `/kaggle/working`; it is
# rendered inline below (and downloadable from the working directory). You
# should be able to click around through the links.
# =============================================================================

from html import escape

from IPython.display import HTML, display

diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    # Isolate the full document in an iframe so its styles don't leak into the notebook.
    display(
        HTML(
            f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
            'width="100%" height="900" style="border:0"></iframe>'
        )
    )
else:
    print("No diagnostics.html - minimal diagnostics (real submission) suppresses it.")


